# Figure 2E | CAST spatial-domain identification

Spatial-domain identification in BASSFISH whole-brain sections using CAST graph embeddings followed by K-means clustering and region-specific refinement.

Check input/output paths and settings before running. Run the notebook from top to bottom with a fresh kernel.

| Workflow | Purpose |
|---|---|
| 1 | Load the BASSFISH AnnData object and generate normalized expression |
| 2 | Build per-sample coordinate and expression dictionaries |
| 3 | Run `CAST_MARK` to obtain spatial graph embeddings |
| 4 | Identify 20 major spatial domains by K-means |
| 5 | Refine striatal, cortical, cerebellar, hippocampal, and meningeal domains |
| 6 | Export spatial-domain maps and the annotated h5ad object |

The original analysis also contained exploratory second-level refinements and genotype-comparison diagnostics. These were not required for the Figure 2E spatial-domain panel and are not included in the main reproduction workflow below.


## Imports and configuration

The original notebook used absolute server paths. For GitHub release, edit the project-relative paths below. CAST parameters and K-means cluster numbers are retained from the source analysis.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch

import CAST
from CAST import CAST_MARK
from CAST.visualize import kmeans_plot_multiple
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ---------------------------------------------------------------------
# Paths: edit for the local repository/data layout.
# ---------------------------------------------------------------------
DATA_ROOT = Path("data")
OUT_DIR = Path("outputs/Fig2E_CAST_domains")
CAST_OUT_DIR = OUT_DIR / "cast_output"
FIG_DIR = OUT_DIR / "figures"

INPUT_H5AD = DATA_ROOT / "integrated_refined_clustered_250519.h5ad"
OUTPUT_H5AD = OUT_DIR / "integrated_refined_clustered_refined_domain.h5ad"

CAST_OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Major-domain K-means setting from the source notebook.
N_MAJOR_DOMAINS = 20
RANDOM_STATE = 0


## 1. Load BASSFISH data and normalize expression

The source workflow restored the `.raw` matrix and generated a `norm_1e4` layer by normalizing each cell to a total count of 10,000. CAST uses the per-cell spatial coordinates stored in `obs['x']` and `obs['y']` together with this normalized expression matrix.


In [ ]:
adata = ad.read_h5ad(INPUT_H5AD)
adata = adata.raw.to_adata()

required_obs = {"sample", "x", "y"}
missing_obs = required_obs - set(adata.obs.columns)
if missing_obs:
    raise KeyError(f"Missing required obs columns: {sorted(missing_obs)}")

adata.layers["norm_1e4"] = sc.pp.normalize_total(
    adata,
    target_sum=1e4,
    inplace=False,
)["X"]

# Ensure a Scanpy-compatible spatial basis is available for plotting.
if "position" not in adata.obsm:
    adata.obsm["position"] = adata.obs[["x", "y"]].to_numpy()

print(adata)
print("Samples:", sorted(adata.obs["sample"].astype(str).unique()))


## 2. Prepare CAST inputs

CAST is run jointly across all slices. A coordinate matrix and normalized-expression matrix are constructed for each biological sample. Cell IDs are stored in the same sample order so that the concatenated CAST embeddings can later be mapped back to the AnnData rows exactly.


In [ ]:
samples = np.unique(adata.obs["sample"].astype(str))

coords_raw = {
    sample: adata.obs.loc[
        adata.obs["sample"].astype(str) == sample,
        ["x", "y"],
    ].to_numpy()
    for sample in samples
}

exp_dict = {
    sample: adata[
        adata.obs["sample"].astype(str) == sample
    ].layers["norm_1e4"]
    for sample in samples
}

coords_raw_index = {
    sample: adata.obs.loc[
        adata.obs["sample"].astype(str) == sample,
        ["x", "y"],
    ]
    for sample in samples
}

sorted_cell_ids = []
for sample in samples:
    sorted_cell_ids.extend(coords_raw_index[sample].index.tolist())

pd.DataFrame({"cell_id": sorted_cell_ids}).to_csv(
    CAST_OUT_DIR / "sorted_cell_ids.csv",
    index=False,
)

print("Cells prepared for CAST:", len(sorted_cell_ids))


## 3. CAST graph embedding

`CAST_MARK` is applied using the default model settings used in the source notebook. The returned dictionary contains one graph embedding tensor per sample.

This is the computationally intensive step. The original notebook noted that a reduced `epoch_t` can be used for testing, but the manuscript workflow used the default call shown below.


In [ ]:
embed_dict = CAST_MARK(
    coords_raw,
    exp_dict,
    str(CAST_OUT_DIR),
)

print("CAST embeddings:")
for sample in samples:
    print(sample, tuple(embed_dict[sample].shape))


## 4. Major spatial-domain identification

Cells are reordered to match the sample-wise CAST embedding order. The CAST visualization utility `kmeans_plot_multiple` is then used to partition the graph embeddings into 20 major spatial domains across all samples. Major-domain labels are stored as `major_0` through `major_19`.


In [ ]:
adata = adata[sorted_cell_ids].copy()

adata.obs["cast_domain"] = kmeans_plot_multiple(
    embed_dict,
    samples,
    coords_raw,
    "Step1_clustering",
    str(CAST_OUT_DIR),
    k=N_MAJOR_DOMAINS,
    dot_size=1,
    minibatch=False,
)

adata.obs["cast_domain"] = (
    "major_" + adata.obs["cast_domain"].astype(str)
)

print(adata.obs["cast_domain"].value_counts().sort_index())


In [ ]:
major_domain_path = OUT_DIR / "integrated_refined_clustered_major_domain.h5ad"
adata.write_h5ad(major_domain_path)
print("Saved:", major_domain_path)


### Major-domain spatial maps

The following cell exports the CAST major-domain map for each sample using a common domain palette. These maps are useful for checking reproducibility of the large-scale anatomical organization across slices.


In [ ]:
major_categories = sorted(
    adata.obs["cast_domain"].astype(str).unique(),
    key=lambda x: int(x.split("_")[-1]),
)

major_palette = {
    domain: sc.pl.palettes.default_102[i]
    for i, domain in enumerate(major_categories)
}

for sample in samples:
    temp = adata[adata.obs["sample"].astype(str) == sample].copy()

    sc.pl.embedding(
        temp,
        basis="position",
        color="cast_domain",
        palette=major_palette,
        size=5,
        frameon=False,
        show=False,
        title=str(sample),
    )

    plt.savefig(
        FIG_DIR / f"{sample}_CAST_major_domains.pdf",
        bbox_inches="tight",
    )
    plt.close()


## 5. Region-specific refinement of CAST domains

The 20 major CAST domains were further subdivided in graph-embedding space where additional anatomical resolution was required. Refinement used K-means with a fixed random seed (`random_state=0`). Biological region names were then assigned according to the spatial location of the clusters together with marker-expression checks in the source analysis.

The refinement scheme recorded in the source notebook was:

- `major_3`: meningeal/brain-edge-related region, K=3;
- `major_4`: dorsal striatal region, K=4;
- `major_9`: ventral striatal region, K=2;
- `major_5`, `major_11`, `major_18`: cortex, K=8;
- `major_7`, `major_14`, `major_16`: cerebellum, K=3;
- `major_6`, `major_12`, `major_19`: hippocampal region, K=6.

Other major domains were renamed directly according to their anatomical identity.


In [ ]:
# Concatenate embeddings in the same sample order used to build sorted_cell_ids.
embed_all = torch.cat(
    [embed_dict[sample] for sample in samples],
    dim=0,
)

if embed_all.shape[0] != len(sorted_cell_ids):
    raise ValueError(
        "Embedding rows do not match the number of ordered cells: "
        f"{embed_all.shape[0]} vs {len(sorted_cell_ids)}"
    )

name_to_idx = {
    cell_id: i
    for i, cell_id in enumerate(sorted_cell_ids)
}

adata_new = adata.copy()
adata_new.obs["refined_domain"] = (
    adata_new.obs["cast_domain"].astype(str)
)


In [ ]:
def refine_cast_domains(
    adata,
    source_domains,
    embed_tensor,
    name_to_idx,
    prefix,
    n_clusters,
    source_col="cast_domain",
    target_col="refined_domain",
    random_state=0,
):
    """Subcluster selected CAST domains in the CAST embedding space."""

    target_ids = adata.obs.index[
        adata.obs[source_col].isin(source_domains)
    ].tolist()

    if len(target_ids) == 0:
        raise ValueError(
            f"No cells found for {source_col} in {source_domains}"
        )

    target_indices = [name_to_idx[cell_id] for cell_id in target_ids]
    embed_sub = embed_tensor[
        torch.as_tensor(target_indices, dtype=torch.long)
    ]

    data_np = embed_sub.detach().cpu().numpy()

    km = KMeans(
        n_clusters=n_clusters,
        n_init="auto",
        random_state=random_state,
    ).fit(data_np)

    labels = pd.Series(
        [f"{prefix}_{i}" for i in km.labels_],
        index=target_ids,
    )

    adata.obs[target_col] = adata.obs[target_col].astype(str)
    adata.obs.loc[target_ids, target_col] = labels

    return adata


### 5.1 Initial anatomical renaming and meningeal refinement


In [ ]:
adata_new = refine_cast_domains(
    adata_new,
    source_domains=["major_3"],
    embed_tensor=embed_all,
    name_to_idx=name_to_idx,
    prefix="Meninges1",
    n_clusters=3,
    random_state=RANDOM_STATE,
)

DIRECT_DOMAIN_MAP = {
    "major_0": "WM",
    "major_1": "OLF_1",
    "major_15": "OLF_2",
    "major_2": "OLF_3",
    "major_17": "Meninges2",
    "major_8": "TH",
    "major_13": "VS",
}

adata_new.obs["refined_domain"] = (
    adata_new.obs["refined_domain"]
    .replace(DIRECT_DOMAIN_MAP)
)


### 5.2 Striatal refinement


In [ ]:
# Dorsal striatal CAST domain.
adata_new = refine_cast_domains(
    adata_new,
    source_domains=["major_4"],
    embed_tensor=embed_all,
    name_to_idx=name_to_idx,
    prefix="Dorsal_str",
    n_clusters=4,
    random_state=RANDOM_STATE,
)

adata_new.obs["refined_domain"] = (
    adata_new.obs["refined_domain"].replace({
        "Dorsal_str_0": "Dorsal_STR_1",
        "Dorsal_str_1": "Dorsal_STR_1",
        "Dorsal_str_2": "Dorsal_STR_2",
        "Dorsal_str_3": "NAc",
    })
)

# Ventral striatal CAST domain.
adata_new = refine_cast_domains(
    adata_new,
    source_domains=["major_9"],
    embed_tensor=embed_all,
    name_to_idx=name_to_idx,
    prefix="Ventral_str",
    n_clusters=2,
    random_state=RANDOM_STATE,
)

adata_new.obs["refined_domain"] = (
    adata_new.obs["refined_domain"].replace({
        "Ventral_str_0": "Ventral_STR_ICj_outer",
        "Ventral_str_1": "Ventral_STR",
        "major_10": "Ventral_STR_ICj_inner",
    })
)


### 5.3 Cortical refinement


In [ ]:
CORTEX_MAJOR_DOMAINS = ["major_5", "major_11", "major_18"]

adata_new = refine_cast_domains(
    adata_new,
    source_domains=CORTEX_MAJOR_DOMAINS,
    embed_tensor=embed_all,
    name_to_idx=name_to_idx,
    prefix="Cortex",
    n_clusters=8,
    random_state=RANDOM_STATE,
)

adata_new.obs["refined_domain"] = (
    adata_new.obs["refined_domain"].replace({
        "Cortex_0": "L5",
        "Cortex_1": "L6b",
        "Cortex_2": "L4",
        "Cortex_3": "L4",
        "Cortex_4": "L5",
        "Cortex_5": "L6a",
        "Cortex_6": "L2/3_Wfs1",
        "Cortex_7": "L2/3",
    })
)


### 5.4 Cerebellar refinement


In [ ]:
CEREBELLUM_MAJOR_DOMAINS = ["major_7", "major_14", "major_16"]

adata_new = refine_cast_domains(
    adata_new,
    source_domains=CEREBELLUM_MAJOR_DOMAINS,
    embed_tensor=embed_all,
    name_to_idx=name_to_idx,
    prefix="Cerebellum",
    n_clusters=3,
    random_state=RANDOM_STATE,
)

adata_new.obs["refined_domain"] = (
    adata_new.obs["refined_domain"].replace({
        "Cerebellum_0": "ML",
        "Cerebellum_1": "GCL",
        "Cerebellum_2": "PCL",
    })
)


### 5.5 Hippocampal refinement


In [ ]:
HIPPO_MAJOR_DOMAINS = ["major_6", "major_12", "major_19"]

adata_new = refine_cast_domains(
    adata_new,
    source_domains=HIPPO_MAJOR_DOMAINS,
    embed_tensor=embed_all,
    name_to_idx=name_to_idx,
    prefix="Hippo",
    n_clusters=6,
    random_state=RANDOM_STATE,
)

adata_new.obs["refined_domain"] = (
    adata_new.obs["refined_domain"].replace({
        "Hippo_0": "DG",
        "Hippo_1": "Meninges_hippo",
        "Hippo_2": "CA1",
        "Hippo_3": "DG_Hilus",
        "Hippo_4": "CA1/2",
        "Hippo_5": "CA3",
    })
)


## 6. Diagnostic marker and spatial checks

The source notebook used marker-expression and spatial-distribution plots to support the anatomical interpretation of refined CAST domains. The examples below reproduce the main diagnostic logic without hard-coding a particular manuscript sample.


In [ ]:
def plot_domain_diagnostic(
    adata,
    sample,
    subset_domains=None,
    color="refined_domain",
    size=10,
    cmap="Reds",
):
    temp = adata[adata.obs["sample"].astype(str) == str(sample)].copy()

    if subset_domains is not None:
        temp = temp[
            temp.obs["cast_domain"].isin(subset_domains)
        ].copy()

    kwargs = {}
    if color in temp.obs.columns:
        kwargs["palette"] = None
    else:
        kwargs["cmap"] = cmap

    sc.pl.embedding(
        temp,
        basis="position",
        color=color,
        size=size,
        frameon=False,
        show=True,
        **kwargs,
    )


# Example:
# plot_domain_diagnostic(
#     adata_new,
#     sample="Homo_1",
#     subset_domains=["major_4"],
#     color="refined_domain",
# )


In [ ]:
# Marker sets used in the source notebook for anatomical checks.
CORTEX_MARKERS = [
    "Wfs1", "Cux2", "Rorb", "Fezf2",
    "Hs3st4", "Foxp2", "Ccn2",
]

cortex_markers_available = [
    g for g in CORTEX_MARKERS
    if g in adata_new.var_names
]

# Example diagnostic dot plot:
# sample_name = "Homo_4"
# temp = adata_new[
#     (adata_new.obs["sample"].astype(str) == sample_name)
#     & (adata_new.obs["cast_domain"].isin(CORTEX_MAJOR_DOMAINS))
# ].copy()
#
# sc.pl.dotplot(
#     temp,
#     cortex_markers_available,
#     groupby="refined_domain",
#     vmax=0.2,
# )


## 7. Figure 2E | Refined spatial-domain maps

A common color dictionary is generated from the complete set of refined domains, ensuring that the same anatomical label has the same color in every slice. The cell-level maps below can be exported for all samples and assembled into the final figure layout.


In [ ]:
refined_categories = sorted(
    adata_new.obs["refined_domain"].astype(str).unique()
)

palette_source = sc.pl.palettes.default_102
refined_palette = {
    domain: palette_source[i % len(palette_source)]
    for i, domain in enumerate(refined_categories)
}

for sample in samples:
    temp = adata_new[
        adata_new.obs["sample"].astype(str) == sample
    ].copy()

    sc.pl.embedding(
        temp,
        basis="position",
        color="refined_domain",
        palette=refined_palette,
        size=5,
        frameon=False,
        show=False,
        title=str(sample),
    )

    plt.savefig(
        FIG_DIR / f"{sample}_CAST_refined_domains.pdf",
        bbox_inches="tight",
    )
    plt.close()
